# Understanding the Router Dashboard

Observability is critical for understanding how llm-d routes requests and
where bottlenecks form. This notebook walks through setting up a full
monitoring stack with Prometheus and Grafana, then importing the llm-d
dashboard to visualize key routing metrics in real time.

By the end of this guide, you will have:
- Prometheus scraping metrics from the EPP and vLLM pods
- The Prometheus adapter exposing custom metrics to Kubernetes
- A Grafana dashboard showing queue depth, cache hit rates, latency, and throughput
- Test traffic flowing so you can see the dashboard in action

In [ ]:
# Environment setup
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["MONITORING_NS"] = "monitoring"

print("Namespace:", os.environ["NAMESPACE"])
print("Monitoring namespace:", os.environ["MONITORING_NS"])

## Step 1: Deploy Prometheus

Prometheus collects time-series metrics from all llm-d components. The EPP
exposes metrics on its `/metrics` endpoint, including:

- `epp_queue_depth` - number of requests waiting per replica
- `epp_request_ttft_seconds` - time-to-first-token histogram
- `epp_request_tpot_seconds` - time-per-output-token histogram
- `epp_prefix_cache_hit_ratio` - fraction of prompt tokens found in cache
- `epp_request_duration_seconds` - end-to-end request duration
- `epp_requests_total` - total request count by status

We use the kube-prometheus-stack Helm chart, which bundles Prometheus,
Alertmanager, and Grafana together.

In [ ]:
# Install the kube-prometheus-stack via Helm
!helm repo add prometheus-community https://prometheus-community.github.io/helm-charts
!helm repo update

# Deploy Prometheus + Grafana
!helm upgrade --install kube-prometheus prometheus-community/kube-prometheus-stack \
    --namespace $MONITORING_NS \
    --create-namespace \
    --set grafana.enabled=true \
    --set grafana.adminPassword=llm-d-admin \
    --set prometheus.prometheusSpec.serviceMonitorSelectorNilUsesHelmValues=false \
    --set prometheus.prometheusSpec.podMonitorSelectorNilUsesHelmValues=false

print("\nPrometheus stack deployed.")
print("Grafana default credentials: admin / llm-d-admin")

In [ ]:
# Create a ServiceMonitor to tell Prometheus to scrape the EPP metrics endpoint

service_monitor = """apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: epp-metrics
  namespace: llm-d
  labels:
    release: kube-prometheus
spec:
  selector:
    matchLabels:
      app: epp
  endpoints:
    - port: metrics
      interval: 15s
      path: /metrics
---
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: vllm-metrics
  namespace: llm-d
  labels:
    release: kube-prometheus
spec:
  selector:
    matchLabels:
      app: vllm
  endpoints:
    - port: metrics
      interval: 15s
      path: /metrics
"""

with open("/tmp/service-monitors.yaml", "w") as f:
    f.write(service_monitor)

!kubectl apply -f /tmp/service-monitors.yaml

print("\nServiceMonitors created.")
print("Prometheus will begin scraping EPP and vLLM metrics within 15 seconds.")

## Step 2: Configure the Prometheus Adapter

The Prometheus adapter bridges Prometheus metrics into the Kubernetes
custom metrics API. This is required for HPA autoscaling (see the
Autoscaling notebook), but it is also useful for querying metrics
via `kubectl`.

We configure rules that translate raw Prometheus queries into named
Kubernetes metrics.

In [ ]:
# Deploy the Prometheus adapter with EPP metric rules

adapter_values = """prometheus:
  url: http://kube-prometheus-prometheus.monitoring.svc:9090
rules:
  custom:
    - seriesQuery: 'epp_queue_depth{namespace="llm-d"}'
      resources:
        overrides:
          namespace: {resource: "namespace"}
          pod: {resource: "pod"}
      name:
        matches: "^(.*)"
        as: "epp_queue_depth"
      metricsQuery: 'avg(epp_queue_depth{<<.LabelMatchers>>})'
    - seriesQuery: 'epp_prefix_cache_hit_ratio{namespace="llm-d"}'
      resources:
        overrides:
          namespace: {resource: "namespace"}
          pod: {resource: "pod"}
      name:
        matches: "^(.*)"
        as: "epp_prefix_cache_hit_ratio"
      metricsQuery: 'avg(epp_prefix_cache_hit_ratio{<<.LabelMatchers>>})'
"""

with open("/tmp/adapter-values.yaml", "w") as f:
    f.write(adapter_values)

!helm upgrade --install prometheus-adapter prometheus-community/prometheus-adapter \
    -f /tmp/adapter-values.yaml \
    -n $MONITORING_NS

print("\nPrometheus adapter deployed.")
print("\nVerify custom metrics are available:")
!kubectl get --raw /apis/custom.metrics.k8s.io/v1beta1 | python3 -m json.tool | head -20

## Step 3: Import the llm-d Grafana Dashboard

The llm-d project provides a pre-built Grafana dashboard JSON that
includes panels for all the key routing and inference metrics. We will
import it as a ConfigMap so Grafana picks it up automatically.

In [ ]:
# Import the llm-d observability dashboard into Grafana
# The dashboard is provided in the production stack repo

!kubectl apply -k llm-d-production-stack/config/observability/grafana/ \
    -n $MONITORING_NS

print("\nGrafana dashboard imported.")
print("\nAccess Grafana:")

# Port-forward Grafana for local access
print("Run the following command in a terminal:")
print("  kubectl port-forward svc/kube-prometheus-grafana 3000:80 -n monitoring")
print("\nThen open http://localhost:3000 in your browser.")
print("Navigate to Dashboards > llm-d Router Dashboard.")

## Step 4: Explore Key Dashboard Panels

The llm-d dashboard is organized into several sections:

### Queue Depth per Replica
Shows the number of pending requests at each vLLM replica over time.
This is the primary signal for load-aware routing. A healthy system
should show roughly even queue depths across replicas. If one replica
is consistently higher, it may indicate a routing imbalance or a
slow GPU.

### Cache Hit Rate Over Time
Tracks what fraction of incoming prompt tokens are found in the
prefix cache. Higher hit rates mean lower TTFT. This metric is
especially useful for multi-turn conversation workloads, where
the conversation history is reused across turns.

### Request Latency Percentiles
Shows p50, p90, p95, and p99 for both TTFT and end-to-end latency.
The gap between p50 and p99 reveals tail latency issues. A large gap
often means some requests are hitting cold replicas or contending
for KV-cache space.

### Throughput
Displays requests per second and tokens per second across the cluster.
This helps with capacity planning. If throughput plateaus while queue
depth rises, you need more replicas.

In [ ]:
# Query Prometheus directly to see the metrics the dashboard uses
# This helps you understand what data feeds each panel

import subprocess
import json

def query_prometheus(query):
    """Run a PromQL query via kubectl port-forward."""
    result = subprocess.run(
        ["kubectl", "exec", "-n", "monitoring",
         "deploy/kube-prometheus-prometheus", "--",
         "wget", "-qO-",
         f"http://localhost:9090/api/v1/query?query={query}"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        data = json.loads(result.stdout)
        return data.get("data", {}).get("result", [])
    return []

# Query 1: Queue depth per replica
print("=== Queue Depth per Replica ===")
results = query_prometheus('epp_queue_depth{namespace="llm-d"}')
for r in results:
    pod = r["metric"].get("pod", "unknown")
    value = r["value"][1]
    print(f"  {pod}: {value} requests queued")

print()

# Query 2: Cache hit rate
print("=== Prefix Cache Hit Rate ===")
results = query_prometheus('avg(epp_prefix_cache_hit_ratio{namespace="llm-d"})')
for r in results:
    value = float(r["value"][1])
    print(f"  Average cache hit rate: {value:.2%}")

print()

# Query 3: TTFT p99
print("=== TTFT p99 ===")
results = query_prometheus(
    'histogram_quantile(0.99, rate(epp_request_ttft_seconds_bucket{namespace="llm-d"}[5m]))'
)
for r in results:
    value = float(r["value"][1])
    print(f"  TTFT p99: {value*1000:.0f} ms")

print()

# Query 4: Request throughput
print("=== Throughput ===")
results = query_prometheus(
    'sum(rate(epp_requests_total{namespace="llm-d"}[1m]))'
)
for r in results:
    value = float(r["value"][1])
    print(f"  Requests/sec: {value:.2f}")

## Step 5: Generate Test Traffic

Let us generate some traffic so the dashboard has data to display.
We will use a mix of unique prompts (cache misses) and repeated
prompts (cache hits) to create interesting patterns in the metrics.

In [ ]:
# Generate test traffic and monitor dashboard metrics
import subprocess
import json
import time
import threading

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

print(f"Gateway: {GATEWAY_IP}:8080")
print()

# Define a mix of prompts for varied cache behavior
prompts = [
    # These repeated prompts should produce cache hits
    "Explain what Kubernetes is in two sentences.",
    "Explain what Kubernetes is in two sentences.",
    "What is a container runtime?",
    "What is a container runtime?",
    # These unique prompts should produce cache misses
    "Describe the water cycle in simple terms.",
    "How does photosynthesis work?",
    "What causes tides in the ocean?",
    "Explain the concept of gravity.",
]

request_count = [0]
stop_event = threading.Event()

def send_requests():
    while not stop_event.is_set():
        for prompt in prompts:
            if stop_event.is_set():
                break
            payload = json.dumps({
                "model": "Qwen/Qwen3-32B",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 100,
            })
            subprocess.run(
                ["curl", "-s", f"http://{GATEWAY_IP}:8080/v1/chat/completions",
                 "-H", "Content-Type: application/json", "-d", payload],
                capture_output=True
            )
            request_count[0] += 1

# Start 5 worker threads
threads = [threading.Thread(target=send_requests, daemon=True) for _ in range(5)]
for t in threads:
    t.start()

print("Generating test traffic with 5 concurrent workers...")
print("Watch the Grafana dashboard update in real time.")
print()

# Monitor for 2 minutes
for i in range(8):
    time.sleep(15)
    print(f"  [{(i+1)*15}s] Total requests sent: {request_count[0]}")

stop_event.set()
print(f"\nTraffic generation complete. Total: {request_count[0]} requests.")
print("\nCheck the Grafana dashboard to see:")
print("  - Queue depth spikes and distribution across replicas")
print("  - Cache hit rate increasing as repeated prompts get cached")
print("  - TTFT p50/p99 spread under load")
print("  - Throughput ramp-up and plateau")

In [ ]:
# Snapshot the dashboard metrics after traffic generation
print("=== Post-Traffic Dashboard Snapshot ===")
print()

# Queue depth should have returned to near-zero
print("Queue Depth (should be near zero after traffic stops):")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=epp_queue_depth{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    print(f'  {r["metric"].get("pod", "unknown")}: {r["value"][1]}')
"

print()
print("Cache Hit Rate (should be above 0% if repeated prompts were cached):")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=avg(epp_prefix_cache_hit_ratio{namespace="llm-d"})' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    val = float(r['value'][1])
    print(f'  Average: {val:.2%}')
"

print()
print("Total Requests Served:")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=sum(epp_requests_total{namespace="llm-d"})' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    print(f'  Total: {r["value"][1]}')
"

## Summary

You now have a full observability stack for llm-d:

1. **Prometheus** scrapes metrics from the EPP and vLLM every 15 seconds.
2. **The Prometheus adapter** exposes those metrics to the Kubernetes
   custom metrics API for use by the HPA.
3. **Grafana** provides a pre-built dashboard with four key panel groups:
   queue depth, cache hit rates, latency percentiles, and throughput.

### Key Metrics to Watch

| Metric | Healthy Range | Action if Out of Range |
|--------|--------------|------------------------|
| Queue depth per replica | 0-10 | Scale up replicas or investigate slow pods |
| Cache hit rate | >50% for conversational workloads | Check routing config, ensure prefix-cache scoring is enabled |
| TTFT p99 | <500ms (varies by model) | Check queue depth, consider P/D disaggregation |
| Throughput | Proportional to replica count | Scale up or optimize batch sizes |

### Next Steps

- **Autoscaling**: Use queue depth metrics with the HPA for automatic scaling.
- **Alerting**: Configure Alertmanager rules for SLO violations.
- **Custom panels**: Add model-specific panels if you serve multiple models.